# CIFAR-100 baseline suite (ResNet-18) — SQAKD / DAQAKD / CMT-KD / InstantNet-CDT / Any-Precision / QAT-only

Self-contained for **Kaggle** or **Google Colab**. Runs the same six-method benchmark
already run on TinyImageNet, against **your existing PMABD CIFAR-100 results**.

**Run the cells in order.** Cell 7 is the launcher — one command per technique,
all commented out, so you uncomment the ones you want.

---

### The one thing that determines whether these numbers are usable

Every baseline must start from **the same FP32 CIFAR-100 backbone that your
PMABD run used**. If you train a fresh FP32 model here instead, the baselines
will be measured against a different starting point than PMABD was, and the
comparison is void — this is exactly the mistake that invalidated the first
TinyImageNet comparison (SQAKD's published numbers came from a from-scratch FP32
model at 58.07 while ours was 74.44, and that single difference explained the
entire 16-point gap).

So: **upload your PMABD CIFAR-100 `M_fp32.pth`** in Cell 5. Training a new one is
offered only as a fallback, and it is clearly labelled as producing a
self-consistent but *not* PMABD-comparable table.

### Architecture
`ARCH` in Cell 3 defaults to **resnet18** with a CIFAR stem. Set it to whatever
your PMABD CIFAR-100 run used — if the architectures differ the table is
meaningless.

ResNet is also the better fit for this code than MobileNetV2. Both quantizer
families assume unsigned, ReLU-fed activations; ResNet satisfies that, whereas
MobileNetV2's inverted residuals feed every expand conv from a *signed* linear
bottleneck, which is what forced the BatchNorm-recalibration and
quantize-dequantize fixes on TinyImageNet.

### Protocol (fixed by `pipeline.get_cifar100_dataloaders`, do not change)
- 32x32 native resolution, no upsampling
- train: 90% of the 50k train split, `RandomCrop(32, padding=4)` + `HFlip` + `AutoAugment(CIFAR10)`
- val: the other 10%, clean transform, drives checkpoint selection
- test: the full untouched 10k official test set, reported only
- normalisation: mean (0.5071, 0.4867, 0.4408), std (0.2675, 0.2565, 0.2761)


## 1. Environment check

In [ ]:
import subprocess, sys, os, platform

print("python  :", sys.version.split()[0])
print("platform:", platform.platform())
try:
    import torch
    print("torch   :", torch.__version__, "| cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu     :", torch.cuda.get_device_name(0),
              f"({torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB)")
except ImportError:
    print("torch   : not installed yet (next cell installs it)")

ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")
WORK = "/kaggle/working" if ON_KAGGLE else ("/content" if ON_COLAB else os.getcwd())
print("workdir :", WORK, "| kaggle:", ON_KAGGLE, "| colab:", ON_COLAB)

# A 16 GB card fits everything here comfortably. CMT-KD is the tight one: it
# holds the student plus up to three quantized teachers. Below ~12 GB, drop its
# batch size (see the launcher cell).

## 2. Clone the repo and install dependencies

Only the `baselines/` package, `pipeline.py` and `pipeline_imagenet_patch.py` are
used. The TinyImageNet weights in the repo are ignored here.

In [ ]:
REPO_URL = "https://github.com/adil-mubashir-ch/pmabd-imagenet.git"
BRANCH   = "experiments/w2a2-qat-only"
REPO     = os.path.join(WORK, "pmabd-imagenet")

if not os.path.isdir(REPO):
    # --filter=blob:none skips the ~350 MB of TinyImageNet checkpoints, which
    # are useless for CIFAR-100 and would dominate the clone time.
    subprocess.run(["git", "clone", "--filter=blob:none", "--branch", BRANCH,
                    REPO_URL, REPO], check=True)
else:
    print("repo already present")

os.chdir(REPO)
sys.path.insert(0, REPO)
print("cwd:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyyaml", "tqdm", "numpy", "scipy", "pandas", "pillow"], check=True)
print("deps ok")

## 3. CIFAR-100 config

Mirrors the TinyImageNet config's structure so `baselines/` needs no changes
beyond the loader patch in Cell 4.

In [ ]:
import yaml, textwrap

CIFAR_OUT = os.path.join(REPO, "outputs", "cifar100_baselines")
os.makedirs(os.path.join(CIFAR_OUT, "logs"), exist_ok=True)

ARCH = "resnet18"   # must match your PMABD CIFAR-100 run
                    # options: resnet18 / resnet34 / resnet50 / mobilenet_v2

cfg = {
    "model": {"arch": ARCH},
    "experiment": {
        "name": f"{ARCH}_cifar100",
        "dataset": "cifar100",
        "num_classes": 100,
        "output_dir": CIFAR_OUT,
        "seed": 42,
    },
    "data": {
        "batch_size": 128,     # 32x32 is cheap; 128 keeps the GPU busy
        "num_workers": 2,      # Kaggle/Colab give 2-4 usable cores
        "data_root": os.path.join(REPO, "data"),
        "val_fraction": 0.10,  # must match the PMABD CIFAR-100 run
        "val_seed": 42,        # ditto - this fixes WHICH images are held out
        "image_size": 32,      # native, no upsampling
    },
}
CFG_PATH = os.path.join(REPO, "configs", "cifar100_baselines.yaml")
with open(CFG_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open(CFG_PATH).read())

# val_seed and val_fraction MUST match whatever your PMABD CIFAR-100 run used.
# They decide which 5000 images are held out. If they differ, the baselines are
# selecting checkpoints on images PMABD trained on, and vice versa.

## 4. Sanity check: loaders and model build

CIFAR-100 support is already in the repo (`build_loaders_from_cfg` routes to
`pipeline.get_cifar100_dataloaders`, and `baselines/common/setup.py` reads
`model.arch` and applies the CIFAR stem when `image_size <= 64`), so there is
nothing to patch. This cell just proves both work before anything long runs.

In [ ]:
from baselines.common import setup as bsetup

cfgd = bsetup.load_config(CFG_PATH)
tr, va, te = bsetup.build_loaders(cfgd, batch_size=128, num_workers=2)
xb, yb = next(iter(tr))
print("train batch:", tuple(xb.shape), "labels", int(yb.min()), "-", int(yb.max()))
print("batches: train", len(tr), "| val", len(va), "| test", len(te))
assert xb.shape[-1] == 32, "expected 32x32 CIFAR inputs"
print("loaders OK")

## 5. The FP32 model — read this before running

**Option A (required for a PMABD-comparable table).** Upload your PMABD
CIFAR-100 FP32 checkpoint (and make sure `ARCH` matches it). On Kaggle add it as a dataset and point `UPLOADED_FP32`
at it; on Colab use the file picker.

**Option B (fallback).** Train a fresh FP32 MobileNetV2 here. The resulting table
is internally consistent — all six baselines share one FP32 init — but its
numbers **cannot be placed in the same table as your PMABD CIFAR-100 results**,
because PMABD started from a different model. If you use Option B, report it as a
standalone baseline comparison and say so.

In [ ]:
UPLOADED_FP32 = None   # e.g. "/kaggle/input/pmabd-cifar100/M_fp32.pth"

import shutil, torch
DST = os.path.join(CIFAR_OUT, "M_fp32.pth")

if UPLOADED_FP32 and os.path.isfile(UPLOADED_FP32):
    shutil.copy(UPLOADED_FP32, DST)
    # The plain fine-tune slot is also read by --fp32-source pretrained.
    shutil.copy(UPLOADED_FP32, os.path.join(CIFAR_OUT, "mobilenetv2_tin_pretrained.pth"))
    ck = torch.load(DST, map_location="cpu", weights_only=False)
    n = len(ck.get("state_dict", ck))
    print(f"OPTION A: using your PMABD FP32 checkpoint ({n} tensors) -> {DST}")
    print("Table IS comparable with your PMABD CIFAR-100 results.")
elif os.path.isfile(DST):
    print(f"FP32 checkpoint already present at {DST}")
else:
    print("OPTION B will run: no checkpoint supplied.")
    print("The resulting table will NOT be comparable with your PMABD results.")

In [ ]:
# OPTION B ONLY. Skipped automatically if Cell 5 found a checkpoint.
# ResNet-18 (CIFAR stem) from scratch on CIFAR-100, 32x32, ~150 epochs.
# On a Kaggle T4/P100 this is roughly 35-50 min.

FORCE_TRAIN_FP32 = False

if os.path.isfile(DST) and not FORCE_TRAIN_FP32:
    print("FP32 checkpoint present - skipping training.")
else:
    import torch, torch.nn as nn, time
    from baselines.common import setup as bsetup
    from pipeline_imagenet_patch import load_model_imagenet

    cfgd = bsetup.load_config(CFG_PATH)
    tr, va, te = bsetup.build_loaders(cfgd, batch_size=128, num_workers=2)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.backends.cudnn.benchmark = True

    model = load_model_imagenet(arch=ARCH, pretrained=False,
                                num_classes=100, device=dev,
                                small_input=True).to(dev)
    EPOCHS = 150
    opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9,
                          weight_decay=5e-4, nesterov=True)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)

    best = -1.0
    for ep in range(EPOCHS):
        model.train(); t0 = time.time()
        for x, y in tr:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            crit(model(x), y).backward()
            opt.step()
        sch.step()
        v1, _ = bsetup.evaluate(model, va, dev)
        if v1 > best:
            best = v1
            t1, t5 = bsetup.evaluate(model, te, dev)
            torch.save({"state_dict": model.state_dict(), "best_val_top1": v1,
                        "test_top1": t1, "epoch": ep + 1}, DST)
        if ep % 10 == 0 or ep == EPOCHS - 1:
            print(f"ep {ep+1:3d}/{EPOCHS}  val {v1:5.2f}  best {best:5.2f}  "
                  f"{time.time()-t0:.0f}s", flush=True)

    shutil.copy(DST, os.path.join(CIFAR_OUT, "mobilenetv2_tin_pretrained.pth"))
    ck = torch.load(DST, map_location="cpu", weights_only=False)
    print(f"\nFP32 done: val {ck['best_val_top1']:.2f}  test {ck['test_top1']:.2f}")

## 6. Matched epoch budgets + smoke test

`--schedule matched` gives each baseline the epoch count **your PMABD CIFAR-100
run spent at that precision**. Fill these in from that run's
`logs/stages.json`, otherwise the "matched" comparison is matched to nothing.

If you do not have those counts, use `--schedule paper` in Cell 7 instead and say
so in the write-up — do not leave fabricated values here.

In [ ]:
# EDIT THESE to your PMABD CIFAR-100 per-precision epoch counts.
PMABD_CIFAR_EPOCHS = {
    "w8a8": None,
    "w4a4": None,
    "w3a3": None,
    "w2a2": None,
}

import baselines.common.schedules as S
known = {k: v for k, v in PMABD_CIFAR_EPOCHS.items() if v}
if known:
    S.LADDER_EPOCHS.update(known)
    print("matched budgets set:", known)
    missing = [k for k, v in PMABD_CIFAR_EPOCHS.items() if not v]
    if missing:
        print(f"still unset (use --schedule paper for these): {missing}")
else:
    print("No budgets set. Use --schedule paper in the launcher cell.")
    print("Do NOT invent numbers here - an unmatched 'matched' run is worse "
          "than an honest 'paper' one.")

In [ ]:
# Smoke test: builds each method, checks EVAL-MODE accuracy before any gradient
# step, then runs 30 real steps. At W8A8 quantization is near-lossless, so
# eval@init must land near the FP32 model, not at chance (1.00% for 100 classes).
#
# This is the check that would have caught the bug which invalidated four
# TinyImageNet runs: training looked healthy at ~75% train accuracy while eval
# sat at exactly chance for tens of epochs, because BatchNorm running statistics
# were never recalibrated on the quantized network.

!python -m baselines.smoke_test --config configs/cifar100_baselines.yaml \
    --precision w8a8 --steps 30 --batch-size 64 --num-workers 2

## 7. Launcher — uncomment what you want to run

Each line is one technique at one precision. `stages.json` records finished runs,
so re-running this cell skips whatever already completed; interrupted stages
resume from their last epoch.

Swap `--schedule matched` for `--schedule paper` on any precision whose PMABD
epoch count you did not set in Cell 6.

`qat_only` is the no-distillation control. Run it first if you run nothing else:
without it you cannot tell how much of each method's accuracy comes from
distillation rather than from the quantizer and schedule.

In [ ]:
C = "--config configs/cifar100_baselines.yaml --num-workers 2 --schedule matched"

# ---- QAT-only (no KD) — the control. Cheapest row; run this first. ---------
# !python -m baselines.run_baseline --method qat_only --precision w8a8 {C}
# !python -m baselines.run_baseline --method qat_only --precision w4a4 {C}
# !python -m baselines.run_baseline --method qat_only --precision w3a3 {C}
# !python -m baselines.run_baseline --method qat_only --precision w2a2 {C}

# ---- SQAKD (AISTATS 2024) — KL from a frozen FP32 teacher, no labels -------
# !python -m baselines.run_baseline --method sqakd --precision w8a8 {C}
# !python -m baselines.run_baseline --method sqakd --precision w4a4 {C}
# !python -m baselines.run_baseline --method sqakd --precision w3a3 {C}
# !python -m baselines.run_baseline --method sqakd --precision w2a2 {C}

# ---- DAQAKD (2025) — SQAKD + strong DA. Lower bound, not the full method ---
# !python -m baselines.run_baseline --method daqakd --precision w8a8 {C}
# !python -m baselines.run_baseline --method daqakd --precision w4a4 {C}
# !python -m baselines.run_baseline --method daqakd --precision w3a3 {C}
# !python -m baselines.run_baseline --method daqakd --precision w2a2 {C}

# ---- CMT-KD (WACV 2023) — frozen teachers; add --batch-size 64 under 12 GB -
# !python -m baselines.run_baseline --method cmtkd --precision w8a8 {C}
# !python -m baselines.run_baseline --method cmtkd --precision w4a4 {C}
# !python -m baselines.run_baseline --method cmtkd --precision w3a3 {C}
# !python -m baselines.run_baseline --method cmtkd --precision w2a2 {C}

# ---- InstantNet CDT (DAC 2021) — ONE run serves several precisions ---------
#      w3a3 trains bits [3,4,8,32]; w2a2 trains [2,3,4,8,32] (~25% slower).
# !python -m baselines.run_baseline --method instantnet_cdt --precision w3a3 {C}
# !python -m baselines.run_baseline --method instantnet_cdt --precision w2a2 {C}

# ---- Any-Precision (AAAI 2021) — also one run per bit list -----------------
# !python -m baselines.run_baseline --method any_precision --precision w3a3 {C}
# !python -m baselines.run_baseline --method any_precision --precision w2a2 {C}

print("Nothing ran - every line above is commented out. Uncomment to launch.")

## 8. Results table

In [ ]:
!python -m baselines.aggregate --config configs/cifar100_baselines.yaml \
    --csv results/cifar100_baseline_comparison.csv

import glob, json
print("\n--- raw stage records ---")
for f in sorted(glob.glob("baselines/outputs/*/logs/stages.json")):
    for s in json.load(open(f)):
        pp = s.get("per_precision") or {}
        extra = "  " + " ".join(f"{k}={v['test_top1']}" for k, v in pp.items()) if len(pp) > 1 else ""
        print(f"{s.get('stage_id'):<26} {str(s.get('precision')):<7} "
              f"test={s.get('test_top1')!s:<7} h={round(float(s.get('hours') or 0), 2):<6}"
              f" ep={s.get('epochs_run')}{extra}")

## 9. Reading the table — three traps

**Architecture must match PMABD's.** If your PMABD CIFAR-100 run used a
different backbone from `ARCH`, nothing in this table can be compared with it.

**The switchable methods' FP32 is not your FP32.** InstantNet and Any-Precision
train one shared-weight network whose 32-bit branch is itself being optimised, so
it drifts away from the FP32 model everything started at. On TinyImageNet it fell
to 65.32 and 67.90 against a 75.57 start. Their small FP32→W3A3 drop therefore
does **not** mean they preserve accuracy better — they simply had less to lose.
Report their own FP32 branch in the FP32 row, not the shared starting value.

**CMT-KD here runs with frozen teachers.** Mutual learning and teacher-side
training are disabled for compute reasons. The paper's own ablation loses ~1.2
points without mutual learning, so label the row "CMT-KD (frozen teachers)".

**DAQAKD here is a lower bound.** The Contextual-Mutual-Information policy search
is not reproduced; this is SQAKD under a fixed strong augmentation. Label it
accordingly.

Also worth stating: baselines keep the first conv and classifier at FP32 (their
own convention) while the PMABD ladder holds both at W8A8, which is harder. Pass
`--first-last-bits 8,8` for a sensitivity row if you want them equalised.

### Saving results
On Kaggle, `/kaggle/working` persists as the notebook output. On Colab, copy to
Drive before the runtime recycles:
```python
from google.colab import drive; drive.mount("/content/drive")
!cp -r baselines/outputs /content/drive/MyDrive/cifar100_baselines
```